# M3-B1 — Fusion de deux sources + colonne de provenance

> ~30-40 min. Le geste **exact** que le cas d'usage certif attend côté
> données : réunir **deux exports comparables** (même métier, deux
> origines) dans un seul tableau, **sans perdre d'où vient chaque ligne**.

Ici : deux sites d'Acerox t'envoient chacun un export de capteurs. Même
idée, mais ce **ne sont pas** exactement les mêmes fichiers. Ton travail :
les empiler proprement et repérer ce qui cloche.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('..') / 'data'
a = pd.read_csv(DATA_DIR / 'capteurs_site_A.csv')
b = pd.read_csv(DATA_DIR / 'capteurs_site_B.csv')
print('A', a.shape, '| colonnes', list(a.columns))
print('B', b.shape, '| colonnes', list(b.columns))

A (25, 5) | colonnes ['ts', 'machine_id', 'temp_c', 'vibration_mm_s', 'debit_l_min']
B (22, 5) | colonnes ['ts', 'machine_id', 'temp_c', 'vibration_mm_s', 'firmware']


## 1. Le réflexe naïf (à ne PAS garder)

On empile les deux sans réfléchir. Regarde bien le résultat.

In [2]:
naif = pd.concat([a, b], ignore_index=True)
naif.info()

# TODO — retrouve, dans `naif`, quelles lignes viennent de A et lesquelles
# de B. Tu ne peux pas, hein ? C'est exactement le problème.
print(naif.sample(5, random_state=0))
print("\nAucune colonne ne permet de dire si une ligne vient de site_A ou site_B.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ts              47 non-null     object 
 1   machine_id      47 non-null     object 
 2   temp_c          47 non-null     float64
 3   vibration_mm_s  47 non-null     float64
 4   debit_l_min     25 non-null     float64
 5   firmware        22 non-null     object 
dtypes: float64(3), object(3)
memory usage: 2.3+ KB
                     ts machine_id  temp_c  vibration_mm_s  debit_l_min  \
28  2026-06-01T09:00:00        M06   152.5            2.72          NaN   
33  2026-06-01T14:00:00        M05   152.6            1.84          NaN   
30  2026-06-01T11:00:00        M06   148.5            3.03          NaN   
4   2026-06-01T10:00:00        M03    64.2            1.83        116.3   
18  2026-06-02T00:00:00        M05    66.4            1.99        114.7   

   firmware  
28     v1.3  
33     

> **Ce qui vient de casser** (à formuler toi-même) :
>
> - impossible de dire **d'où vient** chaque ligne ;
> - une colonne (`debit_l_min`) est pleine de `NaN` pour la moitié des
>   lignes — pourquoi ?
> - `temp_c` mélange peut-être **deux échelles** sans prévenir…

## 2. Le bon geste : marquer la provenance AVANT de concaténer

Une colonne `source` = une **fiche d'identité** de chaque ligne. C'est la
base de la traçabilité (RGPD, réconciliation, filtrage par scénario ensuite).

In [3]:
a2 = a.assign(source='site_A')
b2 = b.assign(source='site_B')
fusion = pd.concat([a2, b2], ignore_index=True)
fusion['source'].value_counts()

source
site_A    25
site_B    22
Name: count, dtype: int64

## 3. Maintenant, la provenance te fait VOIR les pièges

Fusionner n'est pas empiler. Trois choses à débusquer avec la colonne `source`.

In [4]:
# Piège A — mêmes machine_id des deux côtés : le « M04 » de A est-il
# le même équipement que le « M04 » de B ? (indice : non)
communs = set(a['machine_id']) & set(b['machine_id'])
print('machine_id présents dans les DEUX sites :', sorted(communs))

# Vérification : ces ID désignent-ils vraiment le même équipement ?
for mid in sorted(communs):
    print(mid, '- site_A temp_c:', a.loc[a['machine_id'] == mid, 'temp_c'].tolist())
    print(mid, '- site_B temp_c:', b.loc[b['machine_id'] == mid, 'temp_c'].tolist())
# Valeurs totalement différentes (~65-70 vs ~150-165) -> ce sont deux
# machines physiquement distinctes qui portent le même code par coïncidence.

# TODO — que faire ? préfixer par la source ? (ex. 'site_A::M04')
print("\nIDs uniques avant préfixage :", fusion['machine_id'].nunique())
fusion.loc[:, 'machine_id'] = fusion['source'] + '::' + fusion['machine_id']
print("IDs uniques après préfixage :", fusion['machine_id'].nunique())
print(sorted(fusion['machine_id'].unique()))

machine_id présents dans les DEUX sites : ['M04', 'M05']
M04 - site_A temp_c: [69.9, 64.6, 67.3, 66.6, 70.1, 69.5, 69.7, 64.6]
M04 - site_B temp_c: [164.1, 148.3, 151.3, 155.2]
M05 - site_A temp_c: [71.5, 66.3, 66.4, 67.5]
M05 - site_B temp_c: [152.6, 160.9, 142.9, 144.9, 156.6, 166.2, 147.8]

IDs uniques avant préfixage : 8
IDs uniques après préfixage : 10
['site_A::M01', 'site_A::M02', 'site_A::M03', 'site_A::M04', 'site_A::M05', 'site_B::M04', 'site_B::M05', 'site_B::M06', 'site_B::M07', 'site_B::M08']


In [5]:
# Piège B — même colonne temp_c, mais l'échelle diffère selon la source.
print(fusion.groupby('source')['temp_c'].describe()[['mean', 'min', 'max']])

# TODO — une des deux sources est en °F (unité oubliée). Laquelle ?
# Sans la colonne `source`, cette anomalie était invisible.
# site_B a une moyenne ~154, hors de toute plage plausible pour un capteur
# industriel en °C -> test de conversion Fahrenheit -> Celsius.
converti_b = (fusion.loc[fusion['source'] == 'site_B', 'temp_c'] - 32) * 5 / 9
print("\nsite_B converti (x-32)*5/9 :")
print(converti_b.describe()[['mean', 'min', 'max']])
print("\nà comparer à site_A (déjà en °C) :", fusion.loc[fusion['source'] == 'site_A', 'temp_c'].mean())

# La conversion rapproche fortement site_B de site_A (même plage ~65-75°C)
# -> on corrige la colonne pour homogénéiser l'unité avant toute analyse.
fusion.loc[fusion['source'] == 'site_B', 'temp_c'] = converti_b
print("\nAprès correction d'unité :")
print(fusion.groupby('source')['temp_c'].describe()[['mean', 'min', 'max']])

              mean    min    max
source                          
site_A   68.544000   64.2   76.6
site_B  153.681818  142.9  166.2

site_B converti (x-32)*5/9 :
mean    67.601010
min     61.611111
max     74.555556
Name: temp_c, dtype: float64

à comparer à site_A (déjà en °C) : 68.544

Après correction d'unité :
            mean        min        max
source                                
site_A  68.54400  64.200000  76.600000
site_B  67.60101  61.611111  74.555556


In [6]:
# Piège C — debit_l_min n'existe que pour un site -> NaN pour l'autre.
print(fusion.groupby('source')['debit_l_min'].apply(lambda s: s.isna().mean()))

# TODO — colonne réellement absente, ou juste pas transmise ? -> question client.
# 100% de NaN côté site_B : ce n'est pas une "valeur manquante" au sens
# capteur en panne, c'est une colonne qui n'existe pas dans le schéma de
# cet export (site_B a `firmware` à la place). Idem, symétriquement,
# pour `firmware` côté site_A.
print()
print(fusion.groupby('source')['firmware'].apply(lambda s: s.isna().mean()))
print("\n-> À documenter comme question client : site_B mesure-t-il le débit")
print("   via un autre système, ou ne le mesure-t-il vraiment pas ?")

source
site_A    0.0
site_B    1.0
Name: debit_l_min, dtype: float64

source
site_A    1.0
site_B    0.0
Name: firmware, dtype: float64

-> À documenter comme question client : site_B mesure-t-il le débit
   via un autre système, ou ne le mesure-t-il vraiment pas ?


## Ce que tu dois en retenir

- **Fusionner ≠ empiler.** Deux fichiers « du même genre » cachent des
  écarts de schéma, d'unité et de clé.
- La colonne **`source` (provenance)** n'est pas cosmétique : c'est elle
  qui rend les écarts **visibles** et traçables — et qui te laissera plus
  tard filtrer par origine (scénarios, RGPD, réconciliation).
- Reporte une phrase dans `identification_sources.md` : *« Les sources X
  et Y ont été réunies avec une colonne `source` ; écarts constatés : … »*